# 05 · Build Silver — Cleansed Transactions

**Layer:** Silver &nbsp;|&nbsp; **Stage:** transformation &nbsp;|&nbsp; **Target:** `banking_silver.txn_clean`

The heavy stage of the pipeline: deduplicate the transaction feed, reject
what cannot be salvaged, normalise currencies, and conform each transaction
against `silver.dim_account`. Runs in parallel with `04_build_silver_dim_account`
but joins its output, so the pipeline fans both silver tasks out from the
quality gate and fans back in at `06_validate_silver`.

Rejected rows are not discarded silently — each carries a `reject_reason`
and is handed to `10_quarantine_bad_records` for triage.

**Applied here**

1. null `txn_id` rejected (unkeyable)
2. switch-retry duplicates collapsed on `txn_id`
3. non-positive amounts rejected (auth holds, reversal glitches)
4. currency codes upper-cased and validated
5. `txn_ts` parsed to a real timestamp, `txn_date` / `txn_hour` derived
6. inner join to `dim_account` — transactions against unknown accounts drop out
7. `amount_inr` derived via a static FX table so gold can aggregate one unit


In [ ]:
from pyspark.sql import SparkSession, functions as F, Window

spark = SparkSession.builder.appName("medallion-silver-txn-clean").enableHiveSupport().getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "16")
spark.conf.set("spark.sql.adaptive.enabled", "true")

RUN_DATE  = "2026-09-03"
BATCH_ID  = f"batch_{RUN_DATE.replace('-', '')}"
BRONZE_DB = "banking_bronze"
SILVER_DB = "banking_silver"

VALID_CURRENCIES = ["INR", "USD", "EUR", "GBP", "AED", "SGD"]
FX_TO_INR = {"INR": 1.0, "USD": 83.2, "EUR": 90.1, "GBP": 105.4, "AED": 22.6, "SGD": 61.8}

bronze = spark.table(f"{BRONZE_DB}.raw_transactions")
print("bronze rows in:", bronze.count())

In [ ]:
# 1. reject unkeyable rows
rejects = bronze.filter(F.col("txn_id").isNull()).withColumn("reject_reason", F.lit("NULL_TXN_ID"))
keyed   = bronze.filter(F.col("txn_id").isNotNull())
print(f"rejected {rejects.count()} row(s): null txn_id")

In [ ]:
# 2. collapse switch-retry duplicates
dedupe_window = Window.partitionBy("txn_id").orderBy(F.col("_ingested_at").desc())

deduped = (
    keyed
    .withColumn("_rn", F.row_number().over(dedupe_window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)
print(f"collapsed {keyed.count() - deduped.count()} duplicate transaction(s)")

In [ ]:
# 3-4. reject non-positive amounts and unknown currencies
normalised = deduped.withColumn("currency", F.upper(F.trim(F.col("currency"))))

bad_amount = normalised.filter(F.col("amount") <= 0) \
    .withColumn("reject_reason", F.lit("NON_POSITIVE_AMOUNT"))
bad_ccy = normalised.filter(~F.col("currency").isin(VALID_CURRENCIES)) \
    .withColumn("reject_reason", F.lit("INVALID_CURRENCY"))

rejects = rejects.unionByName(bad_amount, allowMissingColumns=True) \
                 .unionByName(bad_ccy, allowMissingColumns=True)

valid = normalised.filter((F.col("amount") > 0) & F.col("currency").isin(VALID_CURRENCIES))
print(f"rejected {bad_amount.count()} non-positive, {bad_ccy.count()} invalid-currency row(s)")

In [ ]:
# 5. parse timestamps and derive calendar attributes
timed = (
    valid
    .withColumn("txn_ts", F.to_timestamp(F.col("txn_ts"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("txn_date", F.to_date(F.col("txn_ts")))
    .withColumn("txn_hour", F.hour(F.col("txn_ts")))
    .withColumn("is_night_txn", (F.col("txn_hour") < 6) | (F.col("txn_hour") >= 22))
    .withColumn("direction", F.upper(F.trim(F.col("direction"))))
    .withColumn("channel", F.upper(F.trim(F.col("channel"))))
)

In [ ]:
# 6. conform against the account dimension
dim = spark.table(f"{SILVER_DB}.dim_account").select(
    "account_id",
    F.col("customer_id").alias("acct_customer_id"),
    F.col("branch_code").alias("acct_branch_code"),
    F.col("account_type").alias("acct_type"),
    F.col("risk_category").alias("acct_risk_category"),
    F.col("is_active").alias("acct_is_active"),
)

before = timed.count()
conformed = timed.join(dim, on="account_id", how="inner")
print(f"dropped {before - conformed.count()} transaction(s) against an unknown account")

In [ ]:
# 7. convert to a single reporting currency
fx = F.create_map([F.lit(x) for kv in FX_TO_INR.items() for x in kv])

txn_clean = (
    conformed
    .withColumn("fx_rate_to_inr", fx[F.col("currency")])
    .withColumn("amount_inr", F.round(F.col("amount") * F.col("fx_rate_to_inr"), 2))
    .withColumn("signed_amount_inr",
                F.when(F.col("direction") == "CREDIT", F.col("amount_inr"))
                 .otherwise(-F.col("amount_inr")))
    .withColumn("_batch_id", F.lit(BATCH_ID))
    .withColumn("_run_date", F.lit(RUN_DATE))
    .withColumn("_processed_at", F.current_timestamp())
    .select(
        "txn_id", "account_id", "acct_customer_id", "acct_branch_code", "acct_type",
        "acct_risk_category", "acct_is_active",
        "txn_ts", "txn_date", "txn_hour", "is_night_txn",
        "amount", "currency", "fx_rate_to_inr", "amount_inr", "signed_amount_inr",
        "direction", "channel", "merchant", "country",
        "_batch_id", "_run_date", "_processed_at",
    )
)

In [ ]:
(
    txn_clean.write
    .mode("overwrite")
    .format("parquet")
    .partitionBy("txn_date")
    .saveAsTable(f"{SILVER_DB}.txn_clean")
)

# Rejects are persisted for the quarantine stage to pick up.
(
    rejects.select("txn_id", "account_id", "amount", "currency", "reject_reason", "_batch_id")
    .write.mode("overwrite").format("parquet")
    .saveAsTable(f"{SILVER_DB}.txn_rejects")
)

out = spark.table(f"{SILVER_DB}.txn_clean")
print(f"wrote {SILVER_DB}.txn_clean    ({out.count()} rows)")
print(f"wrote {SILVER_DB}.txn_rejects  ({spark.table(f'{SILVER_DB}.txn_rejects').count()} rows)")

In [ ]:
out.orderBy("txn_ts").show(25, truncate=False)